In [4]:
import cv2            
import glob
import os
import numpy as np

In [5]:
MODE = "croppedframe"
ROOT_DATA_DIR = "../data/Unprocessed/Golf_Ball_Flighted_data"

In [6]:

Locations = os.listdir(ROOT_DATA_DIR)
location_dir = []
location_name = []
for location in Locations:
    location_name.append(location)
    location = os.path.join(ROOT_DATA_DIR, location)
    location_dir.append(location)

In [7]:
location_dir, location_name

(['../data/Unprocessed/Golf_Ball_Flighted_data\\KP_Common_Area',
  '../data/Unprocessed/Golf_Ball_Flighted_data\\KP_Study_Area'],
 ['KP_Common_Area', 'KP_Study_Area'])

In [8]:
location_to_video_name = {}

for location in location_dir:
    video_paths_at_location = glob.glob(os.path.join(location, "*_sink.mp4")) # only one file with *_sink.mp4 per recording
    video_title = [video.replace("_sink.mp4", "") for video in video_paths_at_location]
    location_to_video_name[location] = video_title

The below code makes folder structure like this
```
Processing
└──Golf_Ball_Putting_data
    └── location_name
        └── video_title
            ├── left
                └── undistored_rectified_frames
            ├── right
                └── undistored_rectified_frames
```



In [9]:
def make_folder(folder_name):
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        print(f"Folder '{folder_name}' created.")
    else:
        print(f"Folder '{folder_name}' already exists.")

In [ ]:
for key in location_to_video_name.keys():
    dest = key.replace("Unprocessed", "Processing")
    print(dest)
    make_folder(dest) 
    for video_title in location_to_video_name[key]:

        src_left_video = video_title +"_source.mp4"
        src_right_video = video_title +"_sink.mp4"
                
        dest_left_video = src_left_video.replace("Unprocessed", "Processing")
        make_folder(dest_left_video) 
        dest_right_video = src_right_video.replace("Unprocessed", "Processing")
        make_folder(dest_right_video)

        left_video = cv2.VideoCapture(src_left_video)
        right_video = cv2.VideoCapture(src_right_video)

        # loading and initialising calibration files
        calibration_file = np.load('./stereo_A1_fullframe.npy', allow_pickle=True).item()
        if MODE == "croppedframe":
            calibration_file = np.load('./stereo_A4_cropped.npy', allow_pickle=True).item()
            left_video.read()   # to synchronise

        Kl, Dl, Kr, Dr, R, T, img_size = calibration_file['Kl'], calibration_file['Dl'], calibration_file['Kr'], calibration_file['Dr'], \
                                calibration_file['R'], calibration_file['T'], calibration_file['img_size']
        R1, R2, P1, P2, Q, validRoi1, validRoi2 = cv2.stereoRectify(Kl, Dl, Kr, Dr, img_size, R, T)
        xmap1, ymap1 = cv2.initUndistortRectifyMap(Kl, Dl, R1, P1, img_size, cv2.CV_32FC1)
        xmap2, ymap2 = cv2.initUndistortRectifyMap(Kr, Dr, R2, P2, img_size, cv2.CV_32FC1)

        counter = 0
        while True:
            ret, left_frame = left_video.read()
            ret, right_frame = right_video.read()
            if not ret: 
                break
            
            
            # perform flip, undistort, and stereo-rectify
            left_frame_flipped = cv2.flip(left_frame, -1)
            right_frame_flipped = cv2.flip(right_frame, -1)
            
            left_img_undistorted_rectified = cv2.remap(left_frame_flipped, xmap1, ymap1, cv2.INTER_LINEAR)
            right_img_undistorted_rectified = cv2.remap(right_frame_flipped, xmap2, ymap2, cv2.INTER_LINEAR)
            

            # visualising them for sanity checks
            cv2.imshow("left_img_undistorted_rectified", left_img_undistorted_rectified)
            cv2.imshow("right_img_undistorted_rectified", right_img_undistorted_rectified)
            cv2.waitKey(1)

            # write left_frame and right frame to dest_left_video and dest_right_video respectively
            dest_left_frame_path = os.path.join(dest_left_video, f"frame_left_{counter}.jpeg")
            dest_right_frame_path = os.path.join(dest_right_video, f"frame_right_{counter}.jpeg")
            cv2.imwrite(dest_left_frame_path, left_img_undistorted_rectified)
            cv2.imwrite(dest_right_frame_path, right_img_undistorted_rectified)
            counter+=1
        cv2.destroyAllWindows()

../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area' created.
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area\20250304_t_233610_source.mp4' created.
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area\20250304_t_233610_sink.mp4' created.
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area\20250304_t_233642_source.mp4' created.
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area\20250304_t_233642_sink.mp4' created.
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area\20250304_t_233718_source.mp4' created.
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area\20250304_t_233718_sink.mp4' created.
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area\20250304_t_233819_source.mp4' created.
Folder '../data/Processed/Golf_Ball_Flighted_data\KP_Common_Area\20250304_t_233819_sink.mp4' created.
Folder '../data/Processed/Golf_Ball_Flighted